# One-Stage vs Two-Stage (Final Comparison)

This notebook does the final comparison with locked choices:
- **One-stage winner**: `global_mahalanobis`
- **Two-stage winner**: `CFLOW two-stage` (loaded from cflow outputs)

It reports matched operating points: `FPR_normal = 5%, 10%, 15%, 20%`.


In [ ]:
# Cell 1: Repo sync
import os, sys, subprocess
from pathlib import Path

REPO = Path('/content/FYP-code')
URL = 'https://github.com/spinelessknave8/FYP_code.git'
if not REPO.exists():
    subprocess.check_call(['git','clone',URL,str(REPO)])
else:
    subprocess.check_call(['git','-C',str(REPO),'fetch','origin'])
    subprocess.check_call(['git','-C',str(REPO),'reset','--hard','origin/main'])

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print('cwd:', Path.cwd())
print(subprocess.check_output(['git','-C',str(REPO),'log','--oneline','-n','2'], text=True))


In [ ]:
# Cell 2: Preflight deps
import platform, numpy as np, torch, torchvision, sklearn
print('python:', platform.python_version())
print('numpy:', np.__version__)
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('sklearn:', sklearn.__version__)
print('cuda:', torch.cuda.is_available())


In [ ]:
# Cell 3: Mount Drive + config
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive', force_remount=False)

# ---------- toggles ----------
RUN_ONE_STAGE = True            # run locked one-stage here
ONE_STAGE_FULL_DATA = True      # True => no pilot caps
HARD_RESET_ONE_STAGE = False    # delete prior one-stage final outputs

# ---------- constants ----------
SEED = 42
SPLITS = ['a','b','c','d']
FPR_TARGETS = [0.05, 0.10, 0.15, 0.20]
UNKNOWN_KNOWN_FPR_TARGET = 0.15

# if FULL_DATA=False, these caps are used
NORMAL_TRAIN_MAX = 1200
NORMAL_VAL_MAX = 250
NORMAL_TEST_MAX = 350
KNOWN_TRAIN_PER_CLASS = 180
KNOWN_VAL_PER_CLASS = 60
KNOWN_TEST_PER_CLASS = 100
UNKNOWN_TEST_MAX = 300

sev = Path('/content/drive/MyDrive/datasets/severstal')
assert sev.exists(), f'Missing {sev}'
assert (sev/'train.csv').exists(), 'Missing train.csv'
assert (sev/'train_images').exists(), 'Missing train_images'

OUT = Path('/content/drive/MyDrive/fyp_outputs/one_stage_vs_two_stage_final')
ONE_OUT = OUT / 'one_stage'
TWO_OUT = Path('/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full')
OUT.mkdir(parents=True, exist_ok=True)
ONE_OUT.mkdir(parents=True, exist_ok=True)

if HARD_RESET_ONE_STAGE and ONE_OUT.exists():
    shutil.rmtree(ONE_OUT)
    ONE_OUT.mkdir(parents=True, exist_ok=True)

print('ONE_OUT:', ONE_OUT)
print('TWO_OUT:', TWO_OUT)


In [ ]:
# Cell 4: Run locked one-stage (global_mahalanobis) across all splits
import json, random, time, sys
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, accuracy_score, precision_recall_fscore_support, confusion_matrix
import yaml
from pathlib import Path

sys.path.insert(0, str(Path('severstral-osr/src').resolve()))
from data import collect_single_label_defect_samples, collect_normal_image_paths, stratified_split
from src.models.resnet50 import build_resnet50
from src.models.embedding import EmbeddingExtractor
from src.osr.gaussian_mahalanobis import fit_gaussians, batch_min_mahalanobis

device = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
TF = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImgDS(Dataset):
    def __init__(self, items, class_to_idx=None):
        self.items = items
        self.class_to_idx = class_to_idx
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p,c = self.items[i]
        x = TF(Image.open(p).convert('RGB'))
        if self.class_to_idx is None:
            return x, c
        return x, self.class_to_idx[c]

def tau_from_fpr(normal_scores, fpr_target):
    return float(np.quantile(normal_scores, 1.0 - fpr_target))

one_rows=[]

if RUN_ONE_STAGE:
    for SPLIT in SPLITS:
        t_split0 = time.time()
        print(f'\n===== ONE-STAGE {SPLIT} =====')
        rng = random.Random(SEED)

        base = yaml.safe_load(Path('severstral-osr/configs/default.yaml').read_text())
        split_cfg = yaml.safe_load(Path(f'severstral-osr/configs/split_{SPLIT}.yaml').read_text())
        base.update(split_cfg)
        known_classes = base['known_classes']
        all_classes = [f'Class_{i}' for i in [1,2,3,4]]
        unknown_class = [c for c in all_classes if c not in known_classes][0]

        samples = collect_single_label_defect_samples(str(sev), 'train.csv', 'train_images')
        normal_paths = collect_normal_image_paths(str(sev), 'train.csv', 'train_images')
        known_samples = [s for s in samples if s[1] in known_classes]
        unknown_samples = [s for s in samples if s[1] == unknown_class]

        k_train_all, k_val_all, k_test_all = stratified_split(known_samples, train_ratio=0.7, val_ratio=0.15, seed=SEED)

        by_cls_train = defaultdict(list); by_cls_val = defaultdict(list); by_cls_test = defaultdict(list)
        for p,c in k_train_all: by_cls_train[c].append((p,c))
        for p,c in k_val_all: by_cls_val[c].append((p,c))
        for p,c in k_test_all: by_cls_test[c].append((p,c))

        known_train=[]; known_val=[]; known_test=[]
        for c in known_classes:
            rng.shuffle(by_cls_train[c]); rng.shuffle(by_cls_val[c]); rng.shuffle(by_cls_test[c])
            if ONE_STAGE_FULL_DATA:
                known_train += by_cls_train[c]
                known_val += by_cls_val[c]
                known_test += by_cls_test[c]
            else:
                known_train += by_cls_train[c][:KNOWN_TRAIN_PER_CLASS]
                known_val += by_cls_val[c][:KNOWN_VAL_PER_CLASS]
                known_test += by_cls_test[c][:KNOWN_TEST_PER_CLASS]

        rng.shuffle(normal_paths)
        if ONE_STAGE_FULL_DATA:
            n_train_cut = int(0.7 * len(normal_paths))
            n_val_cut = int(0.85 * len(normal_paths))
            normal_train = normal_paths[:n_train_cut]
            normal_val = normal_paths[n_train_cut:n_val_cut]
            normal_test = normal_paths[n_val_cut:]
        else:
            normal_train = normal_paths[:NORMAL_TRAIN_MAX]
            normal_val = normal_paths[NORMAL_TRAIN_MAX:NORMAL_TRAIN_MAX+NORMAL_VAL_MAX]
            normal_test = normal_paths[NORMAL_TRAIN_MAX+NORMAL_VAL_MAX:NORMAL_TRAIN_MAX+NORMAL_VAL_MAX+NORMAL_TEST_MAX]

        rng.shuffle(unknown_samples)
        unknown_test = unknown_samples if ONE_STAGE_FULL_DATA else unknown_samples[:UNKNOWN_TEST_MAX]

        class_to_idx = {c:i for i,c in enumerate(known_classes)}

        # train classifier (for embeddings/logits)
        t0=time.time()
        model = build_resnet50(num_classes=len(class_to_idx), pretrained=True).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()

        train_loader = DataLoader(ImgDS(known_train, class_to_idx), batch_size=32, shuffle=True, num_workers=2)
        val_loader = DataLoader(ImgDS(known_val, class_to_idx), batch_size=32, shuffle=False, num_workers=2)

        best_val=-1.0; best_state=None
        for ep in range(6):
            model.train()
            for x,y in train_loader:
                x,y = x.to(device), y.to(device)
                opt.zero_grad(); logits=model(x); loss=crit(logits,y); loss.backward(); opt.step()
            model.eval(); va_c=0; va_n=0
            with torch.no_grad():
                for x,y in val_loader:
                    x,y=x.to(device), y.to(device)
                    l=model(x)
                    va_c += (l.argmax(1)==y).sum().item(); va_n += x.size(0)
            va = va_c/max(1,va_n)
            if va>best_val:
                best_val=va
                best_state={k:v.cpu() for k,v in model.state_dict().items()}

        model.load_state_dict(best_state)
        model.to(device).eval()
        train_sec = time.time()-t0

        extractor = EmbeddingExtractor(model).to(device)
        def infer(items):
            loader = DataLoader(ImgDS(items, None), batch_size=32, shuffle=False, num_workers=2)
            embs, logits, raws = [], [], []
            t_infer0 = time.time()
            with torch.no_grad():
                for x,raw in loader:
                    x=x.to(device)
                    l=model(x)
                    e=extractor(x)
                    logits.append(l.cpu().numpy()); embs.append(e.cpu().numpy()); raws += list(raw)
            infer_sec = time.time()-t_infer0
            return np.concatenate(embs), np.concatenate(logits), raws, infer_sec

        NTR, _, _, _ = infer([(p,'normal') for p in normal_train])
        NVAL, LOG_NVAL, _, _ = infer([(p,'normal') for p in normal_val])
        NTEST, LOG_NTEST, _, t_n = infer([(p,'normal') for p in normal_test])
        KTR, _, RAW_KTR, _ = infer(known_train)
        KVAL, LOG_KVAL, RAW_KVAL, _ = infer(known_val)
        KTEST, LOG_KTEST, RAW_KTEST, t_k = infer(known_test)
        UTEST, LOG_UTEST, RAW_UTEST, t_u = infer(unknown_test)

        infer_total_sec = t_n + t_k + t_u
        infer_images = len(NTEST) + len(KTEST) + len(UTEST)
        infer_sec_per_image = infer_total_sec / max(1, infer_images)

        y_ktr = np.array([class_to_idx[r] for r in RAW_KTR], dtype=np.int64)

        # locked one-stage scoring: global Mahalanobis for defect screening
        mu = NTR.mean(axis=0)
        cov = np.cov(NTR, rowvar=False) + 1e-3*np.eye(NTR.shape[1])
        inv = np.linalg.inv(cov)
        score = lambda X: np.sum(((X - mu) @ inv) * (X - mu), axis=1)

        S_n_val = score(NVAL)
        S_n_test = score(NTEST)
        S_k_test = score(KTEST)
        S_u_test = score(UTEST)

        # threshold-free screening metrics
        y_screen = np.concatenate([np.zeros(len(S_n_test)), np.ones(len(S_k_test)+len(S_u_test))])
        s_screen = np.concatenate([S_n_test, S_k_test, S_u_test])
        auroc = float(roc_auc_score(y_screen, s_screen))
        auprc = float(average_precision_score(y_screen, s_screen))

        # unknown branch score (known-class mahal)
        known_params = fit_gaussians(KTR, y_ktr, 1e-3)
        U_VAL = batch_min_mahalanobis(KVAL, known_params)
        U_TEST_K = batch_min_mahalanobis(KTEST, known_params)
        U_TEST_U = batch_min_mahalanobis(UTEST, known_params)
        u_tau = float(np.quantile(U_VAL, 1.0 - UNKNOWN_KNOWN_FPR_TARGET))

        # truth for 3-way
        y3_true = np.concatenate([
            np.zeros(len(NTEST), dtype=np.int64),
            np.ones(len(KTEST), dtype=np.int64),
            np.full(len(UTEST), 2, dtype=np.int64),
        ])

        for fpr_t in FPR_TARGETS:
            tau = tau_from_fpr(S_n_val, fpr_t)
            pred_n_def = S_n_test > tau
            pred_k_def = S_k_test > tau
            pred_u_def = S_u_test > tau

            # 3-way prediction
            y_n = np.where(pred_n_def, 1, 0)
            y_k = np.where(~pred_k_def, 0, np.where(U_TEST_K > u_tau, 2, 1))
            y_u = np.where(~pred_u_def, 0, np.where(U_TEST_U > u_tau, 2, 1))
            y3_pred = np.concatenate([y_n, y_k, y_u])

            acc3 = float(accuracy_score(y3_true, y3_pred))
            bal_acc3 = float(balanced_accuracy_score(y3_true, y3_pred))
            prec, rec, f1, _ = precision_recall_fscore_support(y3_true, y3_pred, average='macro', zero_division=0)
            cm = confusion_matrix(y3_true, y3_pred, labels=[0,1,2]).tolist()

            fpr_normal = float(np.mean(pred_n_def))
            tpr_defect = float(np.mean(np.concatenate([pred_k_def, pred_u_def])))
            tpr_unknown = float(np.mean(y_u == 2))
            fpr_known_as_unknown = float(np.mean(y_k == 2))
            false_alarms_per_100 = 100.0 * fpr_normal
            specificity_normal = 1.0 - fpr_normal

            one_rows.append({
                'method_system':'one_stage_global_mahalanobis',
                'split': f'split_{SPLIT}',
                'fpr_target': fpr_t,
                'AUROC_defect_screening': auroc,
                'AUPRC_defect_screening': auprc,
                'TPR_defect@FPR': tpr_defect,
                'TPR_unknown@FPR': tpr_unknown,
                'FPR_known_as_unknown@FPR': fpr_known_as_unknown,
                'FPR_normal_realized': fpr_normal,
                'Specificity_normal': specificity_normal,
                'FalseAlarms_per_100': false_alarms_per_100,
                'acc_3way': acc3,
                'balanced_acc_3way': bal_acc3,
                'macro_precision_3way': float(prec),
                'macro_recall_3way': float(rec),
                'macro_f1_3way': float(f1),
                'cm_3way': json.dumps(cm),
                'train_sec': train_sec,
                'infer_sec_total_test': infer_total_sec,
                'infer_sec_per_image': infer_sec_per_image,
                'total_split_sec': time.time() - t_split0,
                'notes': f'unknown_known_fpr_target={UNKNOWN_KNOWN_FPR_TARGET}'
            })

    one_df = pd.DataFrame(one_rows)
    one_df.to_csv(ONE_OUT/'one_stage_locked_full_summary.csv', index=False)
    print('Saved:', ONE_OUT/'one_stage_locked_full_summary.csv', 'rows=', len(one_df))
    display(one_df.head())



In [ ]:
# Cell 5: Load CFLOW two-stage summary and harmonize columns
import pandas as pd
import numpy as np
from pathlib import Path

cflow_csv = TWO_OUT / 'cflow_two_stage_summary.csv'
assert cflow_csv.exists(), f'Missing {cflow_csv}. Run cflow.ipynb first.'

two = pd.read_csv(cflow_csv)

# normalize expected columns
rename_map = {
    'TPR_unknown@FPR':'TPR_unknown@FPR',
    'TPR_defect@FPR':'TPR_defect@FPR',
    'FPR_known_as_def@FPR':'FPR_known_as_unknown@FPR',
}
for k,v in rename_map.items():
    if k in two.columns and v not in two.columns:
        two[v] = two[k]

two['method_system'] = 'two_stage_cflow'

# optional runtime merge (if available)
runtime_csv = TWO_OUT / 'cflow_runtime_summary.csv'
if runtime_csv.exists():
    rt = pd.read_csv(runtime_csv)
    two = two.merge(rt, on='split', how='left')
else:
    for c in ['train_sec','infer_sec_total_test','infer_sec_per_image','total_split_sec']:
        if c not in two.columns:
            two[c] = np.nan

two_path = OUT / 'two_stage_cflow_harmonized.csv'
two.to_csv(two_path, index=False)
print('Saved:', two_path)
display(two.head())


In [ ]:
# Cell 6: Merge one-stage and two-stage + compare
import pandas as pd
from pathlib import Path

one_path = ONE_OUT / 'one_stage_locked_full_summary.csv'
assert one_path.exists(), f'Missing {one_path}. Set RUN_ONE_STAGE=True and run Cell 4.'

one = pd.read_csv(one_path)
two = pd.read_csv(OUT / 'two_stage_cflow_harmonized.csv')

keep_cols = [
    'method_system','split','fpr_target','AUROC_defect_screening','AUPRC_defect_screening',
    'TPR_defect@FPR','TPR_unknown@FPR','FPR_known_as_unknown@FPR','FPR_normal_realized',
    'Specificity_normal','FalseAlarms_per_100','acc_3way','balanced_acc_3way','macro_f1_3way',
    'train_sec','infer_sec_per_image','total_split_sec'
]
for c in keep_cols:
    if c not in two.columns:
        two[c] = pd.NA

cmp_df = pd.concat([one[keep_cols], two[keep_cols]], ignore_index=True)
cmp_df.to_csv(OUT / 'one_stage_vs_two_stage_full_comparison.csv', index=False)

print('Per-split rows:')
display(cmp_df.sort_values(['split','fpr_target','method_system']))

summary = (cmp_df.groupby(['method_system','fpr_target'])
           [['AUROC_defect_screening','TPR_defect@FPR','TPR_unknown@FPR','FPR_known_as_unknown@FPR','FPR_normal_realized','acc_3way','balanced_acc_3way','macro_f1_3way','train_sec','infer_sec_per_image','total_split_sec']]
           .agg(['mean','std'])
           .reset_index())
summary.columns = ['_'.join(c).strip('_') for c in summary.columns.values]
summary.to_csv(OUT / 'one_stage_vs_two_stage_mean_std.csv', index=False)

print('Mean/std:')
display(summary)
print('Saved:', OUT / 'one_stage_vs_two_stage_full_comparison.csv')
print('Saved:', OUT / 'one_stage_vs_two_stage_mean_std.csv')


In [ ]:
# Cell 7: Quick plots
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

summary = pd.read_csv(OUT / 'one_stage_vs_two_stage_mean_std.csv')

for metric in ['TPR_defect@FPR_mean','TPR_unknown@FPR_mean','macro_f1_3way_mean']:
    plt.figure(figsize=(8,4))
    for m in summary['method_system'].unique():
        d = summary[summary['method_system']==m].sort_values('fpr_target')
        plt.plot(d['fpr_target'], d[metric], marker='o', label=m)
    plt.xlabel('FPR target')
    plt.ylabel(metric)
    plt.title(metric + ' across operating points')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    outp = OUT / f'plot_{metric}.png'
    plt.savefig(outp, dpi=150)
    plt.show()
    print('Saved:', outp)


## Notes
- This notebook is the final comparison surface for one-stage winner vs two-stage CFLOW.
- If CFLOW CSV is missing, run `severstral-osr/notebooks/cflow.ipynb` first.
- For runtime parity, add `cflow_runtime_summary.csv` in CFLOW output folder (optional merge supported).
